# 02e Estimate: which lag? The lag chosen inside each training window

**Purpose:** In `02d` each block entered with one fixed lag (the newest month known at the cutoff). Would an older month work better? We let the model **choose the lag inside each training window**, exactly as it chooses `alpha`, so that the choice itself is tested out of sample.
**Input:** `../data/processed/asylum_clean.csv` and the three files of `01c_preprocess_extra_data.ipynb`.
**Output:** `../data/processed/backtest_forecasts_lag_choice.csv` and `../output/tables/02e_chosen_lags.csv`.

**Why not try every lag on the backtest and keep the best?** With 13 lags and three blocks, one of them would look good by chance, and the test period cannot catch it any more (it was used in `04_test.ipynb`). This is the problem of Johannes' "one main comparison" point and of notebook 209 (*choose before you test*): the choice must happen inside the training data.

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

np.random.seed(42)

## 1. Load the target and the extra data (as in `02d`)

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")
months = clean.index
y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)
T = len(months)

def load(path):
    data = pd.read_csv(path)
    data["month"] = pd.to_datetime(data["month"])
    return data.set_index("month")

extra = load("../data/processed/extra_indicators_clean.csv").reindex(months)
shares = load("../data/processed/nationality_shares_clean.csv").reindex(months)
intensity = load("../data/processed/conflict_intensity_clean.csv")

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])

## 2. The features at lag $k$

For every block and every lag $k$ (months back from the origin), the same two features as in `02c`: the change in the month $t-k$ and the change over the three months up to $t-k$. For the conflict index, the weights stay those of the origin $t$ (as in `02d`). The smallest lag is the newest month known at the cutoff: 1 for the neighbours and the conflict index, 0 for the Mediterranean routes. The largest is 12.

In [3]:
first_lag = {"neighbours": 1, "Med routes": 0, "conflict": 1}
LAGS = range(0, 13)

def conflict_level(t, k):
    # sum over the top 10 nationalities at origin t of w * I(t - k), weights fixed at t
    w = shares.iloc[t]
    m = months[t] - pd.DateOffset(months=k)
    if w.isna().all() or m not in intensity.index:
        return None
    top = w.nlargest(10)
    return top, intensity.loc[m, top.index]

def features(block, t, k):
    if block == "conflict":
        out = []
        for span in [1, 3]:
            a, b = conflict_level(t, k), conflict_level(t, k + span)
            if a is None or b is None:
                return np.nan, np.nan
            out.append(float((a[0] * (a[1] - b[1].to_numpy())).sum()))
        return tuple(out)
    s = extra["log_neighbours" if block == "neighbours" else "log_med_routes"].to_numpy(dtype=float)
    if t - k - 3 < 0:
        return np.nan, np.nan
    return s[t - k] - s[t - k - 1], s[t - k] - s[t - k - 3]

F = {(block, k): np.array([features(block, t, k) for t in range(T)])
     for block in first_lag for k in LAGS if k >= first_lag[block]}
print("Feature arrays:", len(F))

Feature arrays: 37


## 3. Exploration before the test period (description only, not a test)

The correlation between the 3-month change of each block at lag $k$ and the change of log applications from $t$ to $t + 2$, after removing the average of each calendar month (so that the common seasonal pattern does not count). With about 140 months, a correlation below about 0.17 in size cannot be told apart from 0 (rule of thumb $2/\sqrt{n}$).

*Not seen in class:* `groupby(...).transform("mean")` gives each row the average of its group.

In [4]:
h = 2
rows_t = list(range(13, last_backtest - h + 1))
target = np.array([log_y[t + h] - log_y[t] for t in rows_t])
calendar = np.array([months[t + h].month for t in rows_t])
target_ds = target - pd.Series(target).groupby(calendar).transform("mean").to_numpy()

corr = pd.DataFrame(index=list(first_lag), columns=list(LAGS), dtype=float)
for (block, k), arr in F.items():
    x = arr[rows_t, 1]
    ok = ~np.isnan(x)
    corr.loc[block, k] = np.corrcoef(x[ok], target_ds[ok])[0, 1]
corr.round(2)

,0,1,2,3,4,5,6,7,8,9,10,11,12
neighbours,NaN,-0.18,-0.08,-0.01,0.02,0.03,0.06,0.12,0.10,0.04,0.03,-0.01,-0.06
Med routes,0.08,-0.07,-0.11,-0.11,-0.12,-0.07,0.01,0.06,0.05,0.05,0.10,0.12,0.06
conflict,NaN,0.01,0.08,-0.01,-0.09,-0.10,-0.06,-0.06,-0.06,-0.00,0.03,-0.00,0.06


**What we see.** Of the 37 correlations, only one is outside the noise band of about 0.17: the neighbours at lag 1 ($-0.18$, and negative, so a rise in the neighbours would mean a smaller Swiss rise two months later). All others lie between $-0.12$ and $+0.12$. With 37 tries, one or two would cross the band by chance alone.

**What this means for us.** No block and no lag shows a clear lead on the Swiss applications two months ahead. This is only a description; the test is the backtest below, where the lag is chosen without looking at the forecast months.

## 4. The backtest: lag and `alpha` chosen together inside each training window

We use Ridge, the model with the lowest MAE in `03_evaluate.ipynb` and in `02d`. At each origin and horizon, every pair (lag, `alpha`) is scored on the same two validation blocks of 12 months inside the training window as in `02b`; the best pair is refitted on the whole training window and gives the forecast. For speed we work with NumPy arrays instead of DataFrames (same numbers). This takes about ten minutes.

In [5]:
ALPHAS = [0.1, 1, 10, 100, 1000, 10000]

def base_features(h):
    # the 14 features of 02b for every origin t (NaN where t < 12 or t + h is outside the data)
    X = np.full((T, 14), np.nan)
    target = np.full(T, np.nan)
    for t in range(12, T - h):
        dummies = np.zeros(11)
        cm = months[t + h].month
        if cm > 1:
            dummies[cm - 2] = 1.0                 # month dummies 2..12, January dropped (as drop_first in 02b)
        X[t] = [log_y[t] - log_y[t - 1], log_y[t - 1] - log_y[t - 2], log_y[t + h - 12] - log_y[t - 12], *dummies]
        target[t] = log_y[t + h] - log_y[t]
    return X, target

def ridge(alpha):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha))

base = {h: base_features(h) for h in range(1, H_MAX + 1)}
rows, chosen = [], []
for block in first_lag:
    lags = [k for k in LAGS if k >= first_lag[block]]
    for h in range(1, H_MAX + 1):
        Xb, target = base[h]
        full = {k: np.hstack([Xb, F[(block, k)]]) for k in lags}
        ok = {k: ~np.isnan(full[k]).any(axis=1) for k in lags}
        first_row = {k: int(np.argmax(ok[k])) for k in lags}
        for t in range(first_origin, last_backtest):
            if t + h > last_backtest:
                continue
            best = (np.inf, None, None)
            for k in lags:
                X, last_row = full[k], t - h                       # last row whose target is known at t
                for alpha in ALPHAS:
                    mse = []
                    for bl in range(2):
                        start = last_row - 24 + 1 + bl * 12
                        valid = np.arange(start, start + 12)
                        train = np.arange(first_row[k], start - h + 1)
                        m = ridge(alpha).fit(X[train], target[train])
                        mse.append(np.mean((target[valid] - m.predict(X[valid])) ** 2))
                    if np.mean(mse) < best[0]:
                        best = (np.mean(mse), k, alpha)
            _, k, alpha = best
            train = np.arange(first_row[k], t - h + 1)
            assert train.max() + h <= t                         # no training target after the origin
            m = ridge(alpha).fit(full[k][train], target[train])
            forecast_log = log_y[t] + m.predict(full[k][[t]])[0]
            rows.append({"model": f"Ridge + {block} (lag chosen)", "block": block, "origin": months[t],
                         "month": months[t + h], "h": h, "forecast_log": forecast_log, "actual_log": log_y[t + h],
                         "forecast": np.exp(forecast_log), "actual": y[t + h], "alpha": alpha, "lag": k})
            chosen.append({"block": block, "h": h, "origin": months[t], "lag": k, "alpha": alpha})

forecasts = pd.DataFrame(rows)
chosen = pd.DataFrame(chosen)
print("Stored forecasts:", forecasts.shape[0])
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())

Stored forecasts: 1260
All forecast months in the backtest (before 2024-10): True
All forecast months after their origin: True


## 5. Which lag gets chosen? ($h = 2$)

If a block really led the applications by some months, the same lag would be chosen again and again. If the chosen lag jumps around, there is no stable lead.

In [6]:
c2 = chosen[chosen["h"] == 2]
summary = c2.groupby("block")["lag"].agg(
    most_chosen=lambda s: s.mode().iloc[0],
    share_of_most_chosen=lambda s: round((s == s.mode().iloc[0]).mean(), 2),
    different_lags="nunique")
print(summary)
print()
print(c2.groupby(["block", "lag"]).size().unstack(fill_value=0))

            most_chosen  share_of_most_chosen  different_lags
block                                                        
Med routes            0                  0.51              12
conflict              2                  0.31              10
neighbours            1                  0.16              11

lag         0   1   2   3   4   5   6   7   8   9   10  11  12
block                                                         
Med routes  72   4   1   5   9   1  18   3  14   7   3   3   0
conflict     0   1  43  22  25  15  11   0   3   7   7   0   6
neighbours   0  22   4   9   1   0   8  19  16   2  21  16  22


**What we see.**
- **Mediterranean routes:** the newest month (lag 0, the UNHCR timing) is chosen at 72 of the 140 origins (51%); the other origins spread over 11 other lags.
- **Conflict index:** lag 2 is chosen most often, but only at 31% of the origins; 10 different lags are chosen.
- **Neighbours:** the choice jumps around: 11 different lags, the most frequent (lag 1) at only 16% of the origins, and lags 10 to 12 almost as often.

**What this means for us.** Only the routes have a lag that is chosen again and again, and it is the newest month, which `02d` already used. For the neighbours and the conflict index there is no stable lead: the chosen lag changes from one window to the next, as expected when a block carries no real signal. Whether the free choice of the lag still improves the forecasts is checked in `03_evaluate.ipynb` (Part B).

## 6. Save

In [7]:
forecasts.to_csv("../data/processed/backtest_forecasts_lag_choice.csv", index=False)
chosen.to_csv("../output/tables/02e_chosen_lags.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts")

Saved 1260 forecasts


## Summary

- **Method:** for each block, the lag (0 to 12 months back, from the newest month known at the cutoff) and `alpha` are chosen together inside each training window, on the validation blocks of `02b`; Ridge; the same 141 origins; no training target after the origin (`assert`).
- **Exploration:** only 1 of 37 block-lag correlations with the future change of applications is outside the noise band (neighbours at lag 1, $-0.18$).
- **Chosen lags:** stable only for the Mediterranean routes (the newest month, 51% of the origins); unstable for the conflict index (10 different lags) and the neighbours (11).
- **Not seen in class (marked above):** `groupby(...).transform("mean")`.
- **Output:** 1,260 forecasts in `../data/processed/backtest_forecasts_lag_choice.csv`, the chosen lags in `../output/tables/02e_chosen_lags.csv`.